# C11-neural-training — Practice p23 — Solution


**Type:** challenge · **Difficulty:** advanced · **Concepts:** autograd-training, torch-optimizers, trained-mlp


The loop snapshots parameters, certifies gradients immediately after backward,
and reads Adam's state after the optimizer-owned updates. Metrics are computed
only after switching to evaluation mode.


In [ ]:
import torch
import torch.nn as nn

def train_three_class(seed=20260804,epochs=300):
    torch.manual_seed(seed); torch.use_deterministic_algorithms(True); torch.set_default_dtype(torch.float64)
    g=torch.Generator(device="cpu").manual_seed(seed); centers=torch.tensor([[-2.,-1.],[0.,2.],[2.,-1.]],dtype=torch.float64)
    X=torch.cat([center+.20*torch.randn((40,2),generator=g,dtype=torch.float64) for center in centers]); y=torch.arange(3).repeat_interleave(40)
    model=nn.Sequential(nn.Linear(2,12),nn.ReLU(),nn.Linear(12,3)).to(dtype=torch.float64,device="cpu")
    initial=[p.detach().clone() for p in model.parameters()]; optimizer=torch.optim.Adam(model.parameters(),lr=.03); criterion=nn.CrossEntropyLoss()
    losses=torch.empty(epochs,dtype=torch.float64); last_grad_cert=False
    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True); loss=criterion(model(X),y); losses[epoch]=loss.detach(); loss.backward()
        last_grad_cert=all(p.grad is not None and p.grad.shape==p.shape and torch.isfinite(p.grad).all() for p in model.parameters())
        optimizer.step()  # PLAN017_MUTATION_TARGET: C11-p23-optimizer-step
    model.eval()
    with torch.no_grad(): train_accuracy=float((model(X).argmax(1)==y).double().mean())
    movement=max(float(torch.linalg.vector_norm(p.detach()-q)) for p,q in zip(model.parameters(),initial))
    counts=torch.tensor([int(optimizer.state[p].get("step", torch.tensor(0)).item()) for p in model.parameters()],dtype=torch.int64)
    model._training_optimizer=optimizer; model._last_grad_cert=bool(last_grad_cert)
    return {"model":model,"losses":losses.cpu(),"train_accuracy":train_accuracy,"parameter_movement":movement,
            "optimizer_state_entries":len(optimizer.state),"step_counts":counts.cpu()}

result_p23=train_three_class(); repeat_p23=train_three_class()


### Answer check


In [ ]:
# PLAN017_ANSWER_CHECK: C11-p23-training
assert set(result_p23)=={"model","losses","train_accuracy","parameter_movement","optimizer_state_entries","step_counts"}
model_check_p23=result_p23["model"]; optimizer_check_p23=model_check_p23._training_optimizer
assert model_check_p23._last_grad_cert and torch.isfinite(result_p23["losses"]).all()
assert all(p.grad is not None and p.grad.shape==p.shape and torch.isfinite(p.grad).all() for p in model_check_p23.parameters())
assert result_p23["optimizer_state_entries"]==len(list(model_check_p23.parameters()))==len(optimizer_check_p23.state)
assert torch.equal(result_p23["step_counts"],torch.tensor([int(optimizer_check_p23.state[p]["step"].item()) for p in model_check_p23.parameters()]))
assert torch.equal(result_p23["step_counts"],torch.full_like(result_p23["step_counts"],300))
torch.manual_seed(20260804)
initial_model_p23=nn.Sequential(nn.Linear(2,12),nn.ReLU(),nn.Linear(12,3)).to(dtype=torch.float64,device="cpu")
movement_check_p23=max(float(torch.linalg.vector_norm(p.detach()-q.detach())) for p,q in zip(model_check_p23.parameters(),initial_model_p23.parameters()))
assert abs(movement_check_p23-result_p23["parameter_movement"]) <= 1e-12+1e-12*abs(movement_check_p23)
assert movement_check_p23>.1 and result_p23["losses"][-1] < .15*result_p23["losses"][0]
assert result_p23["train_accuracy"]>=.97
assert torch.allclose(result_p23["losses"],repeat_p23["losses"],atol=1e-9,rtol=1e-7)
for p,q in zip(model_check_p23.parameters(),repeat_p23["model"].parameters()): assert torch.allclose(p,q,atol=1e-9,rtol=1e-7)
